# KK1 - Analys av global tv-spelsförsäljning

Dataset: 'vgsales.csv'  Detta är taget från https://www.kaggle.com/datasets/gregorut/videogamesales?utm_source=chatgpt.com som i sin tur är en scrape från vgchartz.com. 
Jag vet inte exakt hur datan samlades in eller vilka källor som användes för varje försäljningssiffra. Det är en begränsning, eftersom olika marknader kan vara olika komplett representerade. I analysen tolkar jag därför resultaten som mönster i detta dataset, inte som absoluta sanningar om alla spel eller hela spelmarknaden.

Tema: global och regional försäljning av tv-spel  
Primära verktyg: pandas, numpy, matplotlib

Datasetet 'vgsales.csv' innehåller information om tv-spel och deras rapporterade försäljning i olika regioner. Kolumnerna visar bland annat spelets namn, plattform, år, genre, publisher och försäljning i Nordamerika, Europa, Japan, övriga regioner och globalt. Försäljningen anges i miljoner exemplar.

En rad verkar representera ett spel på en specifik plattform, inte nödvändigtvis ett unikt spel som helhet. Det betyder att samma titel kan förekomma flera gånger om den har släppts på flera plattformar. Det är viktigt för analysen, eftersom spel som släpps brett kan väga tyngre i datasetet än spel som bara finns på en plattform.

Jag tolkar datasetet som en samling över rapporterad spel­försäljning fram till ungefär 2017. Det verkar främst beskriva fysisk eller traditionellt rapporterad försäljning, och fångar troligen inte modern digital försäljning, mobilspel, free-to-play eller abonnemangsmodeller fullt ut. Därför representerar datan inte hela dagens spelmarknad.



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Inläsning och första inspektion/mekanisk inspektion

Först läser jag in datan och gör en snabb inspektion. Det här steget handlar inte om att dra slutsatser än, utan om att förstå materialets form: antal rader, kolumner, datatyper, saknade värden och rimliga numeriska intervall.

In [ ]:
df = pd.read_csv("data/vgsales.csv")
df.head()

In [ ]:
df.shape #Första antal rader andra kolumner.

In [ ]:
missing_summary = df.isna().sum()
missing_summary
# Var det saknas värden, så lite år och publishers.

In [ ]:
df.duplicated().sum() # Inga dubletter i data set:et.

Den första snabb inspektionen visar att datasetet har drygt 16000 rader och 11 kolumner. De viktigaste kolumnerna är regional och global försäljning. Två kolumner behöver tänka till på: `Year` och `Publisher`, eftersom vissa värden saknas och kanske användas direkt i analysen. Om jag vill göra någon typ av tidsdiagram måste jag komma ihåg att `Year` behöver fungera numeriskt på något sätt.

## 3. Datatvätt



In [ ]:
clean_df = df.copy() # skapar en kopia av orginalet som kan "tvättas"

clean_df["Year"] = pd.to_numeric(clean_df["Year"], errors="coerce").astype("Int64") # gör om det till heltal
clean_df["Decade"] = (clean_df["Year"] // 10 * 10).astype("Int64") # Lägger till kolumnen Decade för att kunna visa årtienden istället.

clean_df[["Name", "Platform", "Year", "Decade"]].head()
#

In [ ]:
region_cols = ["NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales"]  # Regionkolumnerna som tillsammans bör motsvara global försäljning.

clean_df["Regional_Total"] = clean_df[region_cols].sum(axis=1)  # Summerar regionerna rad för rad.
clean_df["Sales_Diff"] = clean_df["Global_Sales"] - clean_df["Regional_Total"]  # Räknar skillnaden mellan global och regional summa.
clean_df["Diff_OK"] = np.isclose(clean_df["Global_Sales"], clean_df["Regional_Total"], atol=0.03)  # Tillåter små avrundningsskillnader. Kontrollerar rad för rad om skillnaden är tillräckligt liten.

clean_df[["Global_Sales", "Regional_Total", "Sales_Diff", "Diff_OK"]].describe()


In [ ]:
clean_df["Diff_OK"].value_counts() #Räknar upp alla rader som är true och false om dom,
#hamnar inom ramen för vad som är tillåtet sannolikt att alla skillnader är avrundingar

Jag dubbelkollar att 'Global_Sales' verkar vara summan av regionerna. Skillnaderna är bara upp till 0.02 miljoner exemplar, vilket troligen beror på avrundning. Därför använder jag 'Global_Sales' vidare i analysen.

In [ ]:
analysis_df = clean_df.dropna(subset=["Year", "Publisher"]).copy()

rows_removed = len(clean_df) - len(analysis_df)
rows_removed, len(analysis_df)

In [ ]:
print(f"Antal borttagna rader: {rows_removed}")
print(f"Antal rader kvar: {len(analysis_df)}")

För analysdelen tar jag bort rader där 'Year', 'Genre' eller 'Publisher' saknas. Jag gör det i en separat analys-DataFrame, att arbeta med. Motiveringen är att tidsanalys, genreanalys och publisheranalys kräver dessa fält "rena". Eftersom antalet borttagna rader är litet jämfört med hela datasetet så borde det inte förändra huvudmönstren. Jag blev lite förvirrad först att det blev en annan summa bortagna än rader som saknade värde
men vissa som saknade Year saknade även  publisher.

# Utforskning genom visualisering

Här använder jag olika diagramtyper beroende av frågan jag ställer om datan, och väljer diagram efter vad betraktaren ska kunna jämföra.


### Fråga 1: Vilka genrer dominerar global försäljning?

Här vill jag rangordna kategorier. Därför väljer jag ett horisontellt stapeldiagram. Det gör det lätt att jämföra genrer och läsa etiketterna.

In [ ]:
genre_sales = (
    analysis_df.groupby("Genre", as_index=False)["Global_Sales"]
    .sum()
    .sort_values("Global_Sales", ascending=True)
)

action_sports_sales = genre_sales.loc[
    genre_sales["Genre"].isin(["Action", "Sports"]),
    "Global_Sales"
].sum()

total_global_sales = genre_sales["Global_Sales"].sum()
action_sports_share = action_sports_sales / total_global_sales * 100

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(genre_sales["Genre"], genre_sales["Global_Sales"], color="#4CA860")

ax.set_title(f"Action och Sports står för {action_sports_share:.1f}% av den globala försäljningen")
ax.set_xlabel("Global försäljning (miljoner exemplar)")
ax.set_ylabel("Genre")
ax.set_xlim(left=0)

plt.tight_layout()
plt.show()

Diagrammet visar att global försäljning är ojämnt fördelad mellan genrer. Action och Sports ligger tydligt högst, medan Strategy och Adventure ligger betydligt lägre.

En viktig begränsning är att genre-kolumnen bara verkar ha en genre per spel. Många spel kan passa i flera genrer, men datasetet tvingar in varje spel i en huvudkategori. Det kan göra att breda genrer som Action samlar många olika typer av spel, medan smalare genrer ser mindre ut.

Min tolkning är därför inte bara att “Action säljer mest”, utan att spel med action-, shooter- eller rollspelselement verkar vara starka i datasetet. Sports sticker också ut som en stor egen kategori, troligen eftersom sportspel ofta är tydligare avgränsade som genre.

En annan viktig begränsning är att en rad verkar vara ett spel på en specifik plattform, inte ett unikt spel som helhet. Samma spel kan därför förekomma flera gånger om det släppts på flera plattformar. Det gör att spel och genrer som ofta fungerar på många plattformar kan få större total försäljning i analysen. Sportspel är ett tydligt exempel, eftersom samma årliga titel ofta släpps på flera konsoler och ibland även PC.

### Fråga 2: Global försäljning över tid i datasetet?

Jag väljer linjediagram eftersom frågan handlar om utveckling over tid.

In [ ]:
yearly_sales = (
    analysis_df.groupby("Year", as_index=False)["Global_Sales"]
    .sum()
    .sort_values("Year")
)

peak_year = yearly_sales.loc[yearly_sales["Global_Sales"].idxmax()]

fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(yearly_sales["Year"], yearly_sales["Global_Sales"], color="#F58518", linewidth=2)
ax.scatter(peak_year["Year"], peak_year["Global_Sales"], color="#D62728", zorder=3)

ax.set_title("Nedgången efter 2008 visar troligen datasetets begränsning")
ax.set_xlabel("År")
ax.set_ylabel("Global försäljning (miljoner exemplar)")
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.show()

Diagrammet visar att försäljningen i datasetet toppar runt 2008 och sedan sjunker kraftigt. Jag tolkar inte det som att spelmarknaden faktiskt krympte på samma sätt. Eftersom spelmarknaden fortsatt växa globalt tyder nedgången snarare på att datasetet blir mindre komplett för senare år.

### Fråga 3: Vilka genrer skiljer sig mest mellan Nordamerika, Europa och Japan?



In [ ]:
region_cols = ["NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales"]

region_genre = analysis_df.groupby("Genre")[region_cols].sum()

top_genres = (
    region_genre.sum(axis=1)
    .sort_values(ascending=False)
    .head(6)
    .index
)

region_genre_grouped = region_genre.loc[top_genres].copy()
region_genre_grouped.loc["Other"] = region_genre.drop(top_genres).sum()

plot_df = region_genre_grouped.T

plot_df = plot_df.rename(index={
    "NA_Sales": "Nordamerika",
    "EU_Sales": "Europa",
    "JP_Sales": "Japan",
    "Other_Sales": "Övriga regioner"
})

# Sortera genrer efter total storlek, men lägg restkategorin Other överst i stapeln.
genre_order = plot_df.drop(columns="Other").sum().sort_values(ascending=False).index.tolist()
genre_order.append("Other")

plot_df = plot_df[genre_order]

fig, ax = plt.subplots(figsize=(10, 6))

plot_df.plot(
    kind="bar",
    stacked=True,
    ax=ax
)

ax.set_title("Action, Sports och Shooter bygger störst del av regionernas försäljning förutom i Japan")
ax.set_xlabel("Region")
ax.set_ylabel("Försäljning (miljoner exemplar)")
ax.set_ylim(bottom=0)
ax.legend(title="Genre", bbox_to_anchor=(1.05, 1), loc="upper left")

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

Diagrammet visar att Nordamerika, Europa och övriga regioner har ganska liknande genreprofil i försäljningen. Action, Sports och Shooter tar stor plats i alla tre. Japan sticker däremot ut: Role-Playing tar en mycket större andel där, framför allt på bekostnad av Shooter.

Eftersom diagrammet använder absoluta tal syns också att Nordamerika står för klart störst försäljning i datasetet. Det kan betyda att den nordamerikanska marknaden faktiskt är störst i datan, men det kan också tyda på att datasetet är mer komplett eller mer inriktat på nordamerikansk försäljning. Därför tolkar jag regionala skillnader försiktigt.

In [ ]:
region_cols = ["NA_Sales", "EU_Sales", "JP_Sales"]

region_genre = analysis_df.groupby("Genre")[region_cols].sum()

top_genres = (
    region_genre.sum(axis=1)
    .sort_values(ascending=False)
    .head(6)
    .index
    .tolist()
)

region_genre_grouped = region_genre.loc[top_genres].copy()
region_genre_grouped.loc["Other"] = region_genre.drop(top_genres).sum()

genre_order = top_genres + ["Other"]

genre_colors = {
    "Action": "#4C78A8",
    "Sports": "#F58518",
    "Shooter": "#E45756",
    "Role-Playing": "#72B7B2",
    "Platform": "#54A24B",
    "Misc": "#B279A2",
    "Other": "#9D9D9D",
}

region_names = {
    "NA_Sales": "Nordamerika",
    "EU_Sales": "Europa",
    "JP_Sales": "Japan",
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, region in zip(axes, region_cols):
    values = region_genre_grouped.loc[genre_order, region]
    colors = [genre_colors[genre] for genre in genre_order]

    ax.pie(
        values,
        labels=genre_order,
        autopct="%1.0f%%",
        startangle=90,
        colors=colors
    )
    ax.set_title(region_names[region])

plt.suptitle("Genreandelar per region")
plt.tight_layout()
plt.show()

Jag behåller också pie charts för att visa andelen genrer inom varje region. De gör det lätt att se regionernas interna genreprofil, till exempel att Role-Playing tar större plats i Japan.

Däremot visar pie charts inte relationen mellan regionernas totala försäljning. En lika stor tårtbit i Japan och Nordamerika betyder alltså inte lika många sålda spel, eftersom regionernas totala försäljning skiljer sig mycket. Därför använder jag stacked bar chart för absoluta nivåer och pie charts för andelar.

Misc är en genre i datasetet och betyder ungefär blandade eller svårkategoriserade spel. Den ska inte blandas ihop med Other, som jag själv skapar i diagrammet för att samla mindre genrer.

### Fråga 4: Hur ser topp 10 ute bland de flest sålda spel ut globalt?

Jag väljer stapeldiagram för topp 10 eftersom frågan är väldigt enkel handlar om att jamföra namngivna spel och upptäcka outliers.

In [ ]:
top_games = analysis_df.nlargest(10, "Global_Sales").sort_values("Global_Sales", ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_games["Name"], top_games["Global_Sales"])
ax.set_title("Wii Sports är överlägset störst bland topp 10-spelen globalt sålda i datasetet.")
ax.set_xlabel("Global forsäljning (miljoner exemplar)")
ax.set_ylabel("Spel")
ax.set_xlim(left=0)

plt.show()

Wii Sports sticker ut tydligt jämfört med resten av topp 10. En möjlig förklaring är att spelet följde med Wii-konsolen i många marknader. Det gör att försäljningen inte är helt jämförbar med spel som såldes separat.

### Fråga 5: Är Nordamerika och Europa intresserade av samma spel?

Här vill jag undersöka om de två största marknaderna i datasetet verkar ha liknande smakmönster. För att inte bara jämföra absoluta försäljningssiffror jämför jag varje spels andel av försäljningen i Nordamerika med samma spels andel av försäljningen i Europa.

Jag använder en scatterplot eftersom varje punkt kan representera ett spel. Om punkterna följer en tydlig diagonal trend tyder det på att spel som tar stor plats i den ena regionen ofta också tar stor plats i den andra.

In [ ]:
scatter_df = analysis_df[analysis_df["Global_Sales"] >= 1].copy()

fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(
    scatter_df["NA_Sales"],
    scatter_df["EU_Sales"],
    alpha=0.45,
    s=35,
    color="#4C78A8"
)

ax.set_title("Nordamerika och Europa verkar ha liknande köpmönster")
ax.set_xlabel("Försäljning i Nordamerika (miljoner exemplar)")
ax.set_ylabel("Försäljning i Europa (miljoner exemplar)")
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.show()

In [ ]:
scatter_df = analysis_df[analysis_df["Global_Sales"] >= 1].copy()

fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(
    scatter_df["NA_Sales"],
    scatter_df["EU_Sales"],
    alpha=0.45,
    s=35,
    color="#4C78A8"
)

max_value = max(scatter_df["NA_Sales"].max(), scatter_df["EU_Sales"].max())

ax.plot(
    [0, max_value],
    [0, max_value],
    linestyle="--",
    color="gray",
    linewidth=1,
    label="Lika mycket i båda regioner"
)

ax.set_title("Nordamerika och Europa följer varandra, men Nordamerika ligger oftast högre")
ax.set_xlabel("Försäljning i Nordamerika (miljoner exemplar)")
ax.set_ylabel("Försäljning i Europa (miljoner exemplar)")
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
taste_df = analysis_df.copy()

taste_df["NA_Share"] = taste_df["NA_Sales"] / taste_df["NA_Sales"].sum()
taste_df["EU_Share"] = taste_df["EU_Sales"] / taste_df["EU_Sales"].sum()

taste_df = taste_df[taste_df["Global_Sales"] >= 1].copy()

fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(
    taste_df["NA_Share"] * 100,
    taste_df["EU_Share"] * 100,
    alpha=0.45,
    s=35,
    color="#4C78A8"
)

max_value = max(
    (taste_df["NA_Share"] * 100).max(),
    (taste_df["EU_Share"] * 100).max()
)

ax.plot(
    [0, max_value],
    [0, max_value],
    linestyle="--",
    color="gray",
    linewidth=1,
    label="Samma andel i båda regioner"
)

ax.set_title("Nordamerika och Europa ger ofta liknande vikt åt samma spel")
ax.set_xlabel("Andel av Nordamerikas försäljning (%)")
ax.set_ylabel("Andel av Europas försäljning (%)")
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
ax.legend()

plt.tight_layout()
plt.show()

Jag valde att testa tre versioner av scatterplotten eftersom de visar olika saker.

I den första grafen får x- och y-axeln anpassas efter respektive regions försäljning. Då syns sambandet mellan Nordamerika och Europa tydligt, men grafen döljer delvis att Nordamerika har högre total försäljning i datasetet.

I den andra grafen använder jag samma skala på båda axlarna och lägger till en diagonal jämförelselinje. Då blir det tydligare att många spel säljer mer i Nordamerika än i Europa. Samtidigt kan den grafen bli lite missvisande om frågan handlar om smak, eftersom den jämför absoluta antal sålda exemplar och inte hur viktiga spelen är inom varje region.

I den tredje grafen jämför jag därför varje spels andel av regionens totala försäljning. Det gör den bättre för frågan om smakmönster, eftersom Nordamerikas större totalförsäljning inte får styra lika mycket. Om ett spel ligger nära diagonalen betyder det att spelet har ungefär samma relativa betydelse i båda regionerna.

# Avslutning/Slutsats

Den tydligaste insikten är att datasetet har starka mönster, men också tydliga begränsningar. Action, Sports och Shooter står för en stor del av den globala försäljningen, men genreindelningen behöver tolkas försiktigt. Varje spel verkar bara ha en huvudgenre, trots att många spel egentligen blandar flera genrer. Det gör att breda kategorier som Action kan samla många olika typer av spel.

Tidsdiagrammet visar en topp runt 2008 och sedan en nedgång. Jag tolkar inte det som att spelmarknaden faktiskt krympte så kraftigt. Snarare verkar datasetet bli mindre komplett för senare år, särskilt eftersom digital försäljning, mobilspel och nya affärsmodeller inte verkar fångas fullt ut. Det förstärks av att datan slutar runt 2017.

Regionalt visar datan att Nordamerika står för mycket större försäljning i absoluta tal. Det kan spegla en större marknad, men också att datasetet är mer komplett eller mer fokuserat på Nordamerika. När jag jämför andelar blir det tydligt att Nordamerika och Europa verkar ha ganska liknande smakmönster, medan Japan sticker ut mer, framför allt genom att Role-Playing tar större plats.

En annan viktig begränsning är att en rad verkar vara ett spel på en specifik plattform. Samma spel kan därför förekomma flera gånger om det släppts på flera plattformar. Det kan gynna spel och genrer som ofta släpps brett, till exempel sportspel.

Det datan inte kan berätta är varför spelen sålde bra, om spelarna tyckte om dem, hur lönsamma de var eller hur dagens spelmarknad ser ut. För en starkare analys hade jag velat ha nyare data, digital försäljning, mobilspel, information om bundles, pris, lanseringsland och om varje rad gäller ett unikt spel eller en specifik plattformsversion.